# 416. Partition Equal Subset Sum

## Topic Alignment
- This 0/1 knapsack variant appears in resource allocation problems, such as load balancing tasks across processors, partitioning datasets for distributed training, or optimizing memory allocation in ML pipelines.

## Metadata 摘要
- Source: https://leetcode.com/problems/partition-equal-subset-sum/
- Tags: Dynamic Programming, Array, 0/1 Knapsack
- Difficulty: Medium
- Priority: High

## Problem Statement 原题描述
Given an integer array `nums`, return `true` if you can partition the array into two subsets such that the sum of the elements in both subsets is equal.

**Constraints**:
- 1 <= nums.length <= 200
- 1 <= nums[i] <= 100

## Progressive Hints
- Hint 1: If the total sum is odd, we cannot partition into two equal subsets.
- Hint 2: The problem reduces to: can we find a subset that sums to total_sum / 2?
- Hint 3: This is the classic 0/1 knapsack problem where target is sum/2.
- Hint 4: Use dp[i] to represent whether we can achieve sum i using the numbers.
- Hint 5: For each number, we decide to include it or not. Traverse from right to left to avoid using the same element twice.

## Solution Overview
This is a classic **0/1 Knapsack** problem. The key insight is:
- If we can partition into two equal subsets, each subset sum = total_sum / 2
- Problem becomes: can we select elements that sum to target = total_sum / 2?

**Approaches**:
1. **2D DP**: `dp[i][j]` = can we achieve sum j using first i elements
2. **1D DP (optimized)**: `dp[j]` = can we achieve sum j
3. **Key difference from complete knapsack**: Each element can only be used once, so we traverse from right to left

## Detailed Explanation

### Understanding 0/1 Knapsack

In the **0/1 Knapsack** problem:
- Each item can be used **at most once**
- We decide for each item: include it (1) or exclude it (0)

**Key Principle**: To avoid using the same element twice in 1D DP, **traverse the capacity from right to left**.

---

### Why Traverse Right to Left?

Consider `dp[j]` = can we achieve sum j?

**Wrong (left to right)**:
```python
for num in nums:
    for j in range(num, target + 1):  # Left to right
        dp[j] = dp[j] or dp[j - num]
```
- When we update `dp[j]`, we might use the already-updated `dp[j-num]` in this iteration
- This means we could use the same `num` multiple times!

**Correct (right to left)**:
```python
for num in nums:
    for j in range(target, num - 1, -1):  # Right to left
        dp[j] = dp[j] or dp[j - num]
```
- When we update `dp[j]`, we use `dp[j-num]` from the **previous iteration**
- This ensures each number is used at most once

---

### Step-by-Step Algorithm

**Step 1**: Calculate total sum
- If odd, return False immediately
- Otherwise, target = total_sum / 2

**Step 2**: Initialize DP array
- `dp[0] = True` (we can always achieve sum 0 by selecting nothing)
- `dp[i] = False` for i > 0

**Step 3**: For each number in nums
- For each sum j from target down to num
- If `dp[j - num]` is True, then `dp[j]` can also be True
- Update: `dp[j] = dp[j] or dp[j - num]`

**Step 4**: Return `dp[target]`

---

### Example Walkthrough

**Input**: nums = [1, 5, 11, 5], total_sum = 22, target = 11

**Initial**: `dp = [True, False, False, ..., False]` (size 12)

**After num = 1**:
- j=11 to 1: dp[1] = dp[0] = True
- `dp = [True, True, False, False, ..., False]`

**After num = 5**:
- j=11 to 5: dp[6] = dp[1] = True, dp[5] = dp[0] = True
- `dp = [True, True, False, False, False, True, True, False, ...]`

**After num = 11**:
- j=11: dp[11] = dp[0] = True
- `dp = [True, True, False, ..., True]`

**After num = 5** (second 5):
- dp[11] remains True

**Result**: dp[11] = True, return True

## Complexity Trade-off Table
| Approach | Time | Space | Notes |
| --- | --- | --- | --- |
| Brute force (backtracking) | O(2^n) | O(n) | Try all subsets |
| 2D DP | O(n × sum) | O(n × sum) | dp[i][j] for i items, sum j |
| 1D DP | O(n × sum) | O(sum) | Optimized space, most common |
| Bit manipulation | O(sum × n/64) | O(sum) | Can use bitset for optimization |

In [ ]:
class Solution:
    def canPartition(self, nums: list[int]) -> bool:
        """
        0/1 Knapsack DP solution.
        
        Time: O(n × sum/2) = O(n × sum)
        Space: O(sum/2) = O(sum)
        """
        total_sum = sum(nums)
        
        # If total sum is odd, cannot partition into two equal subsets
        if total_sum % 2 == 1:
            return False
        
        target = total_sum // 2
        
        # dp[j] = can we achieve sum j?
        dp = [False] * (target + 1)
        dp[0] = True  # Base case: sum 0 is always achievable
        
        # For each number (0/1 knapsack: each item used at most once)
        for num in nums:
            # Traverse from right to left to avoid using same element twice
            for j in range(target, num - 1, -1):
                # If we can achieve sum (j - num), we can achieve sum j
                dp[j] = dp[j] or dp[j - num]
        
        return dp[target]

In [ ]:
# Test cases
tests = [
    ([1, 5, 11, 5], True),         # [1,5,5] and [11]
    ([1, 2, 3, 5], False),         # Cannot partition
    ([2, 2, 1, 1], True),          # [2,2] and [1,1]
    ([1], False),                  # Single element
    ([1, 1], True),                # [1] and [1]
    ([100], False),                # Odd sum
    ([1, 2, 5], False),            # 8 is odd
    ([2, 2, 3, 5], True),          # [2,3,5] = 10, not possible... wait [2,2,3,5] = 12, [5,2,2+3]
]

# Correction to test case
tests[7] = ([2, 2, 3, 5], False)  # sum=12, target=6, can make [2,2,2]? no only 2 twos. [3,3]? only one 3. [5,1]? no 1. False
tests[7] = ([3, 3, 3, 4, 5], True)  # sum=18, target=9, can make [4,5] and [3,3,3]

solver = Solution()
for nums, expected in tests:
    result = solver.canPartition(nums)
    assert result == expected, f"Failed for nums={nums}: got {result}, expected {expected}"
print('All tests passed.')

## Complexity Analysis
- **Time**: O(n × target) where n = len(nums), target = sum(nums) / 2
  - For each of n numbers, we iterate through up to target values
  - In worst case, target can be O(n × max(nums)), so overall O(n^2 × max_value)
- **Space**: O(target) = O(sum(nums))
  - Single DP array of size target + 1
  - Can be further optimized using bitset in some languages

## Edge Cases & Pitfalls
- **Odd total sum**: Immediately return False
- **Single element**: Cannot partition into two non-empty subsets, return False
- **All same values**: If even count, return True; if odd, return False
- **Large sums**: Target can be up to 20,000 (200 elements × 100 max value), ensure DP array is large enough
- **Traversal direction**: MUST traverse right to left for 0/1 knapsack to avoid reusing elements
- **Initialization**: dp[0] = True is crucial; it represents selecting no elements for sum 0
- **Early termination**: Can return True immediately if dp[target] becomes True during iteration

## Follow-up Variants
- **K-way partition**: Partition into k subsets of equal sum (NP-complete for k > 2)
- **Partition with minimum difference**: Find two subsets such that |sum1 - sum2| is minimized
- **Partition with constraints**: Additional constraints like subset size or element ordering
- **Weighted partition**: Elements have both weight and value, partition based on different criteria
- **Online version**: Elements arrive in stream, maintain partition dynamically

## Takeaways
- **0/1 Knapsack Pattern**: Each element can be used at most once, traverse capacity from right to left in 1D DP
- **Problem Transformation**: Partition into two equal subsets ⇔ Find subset with sum = total/2
- **Optimization**: Can use early termination when dp[target] becomes True
- **Common Template**: This right-to-left traversal pattern applies to all 0/1 knapsack problems
- **State Definition**: dp[j] represents "can we achieve sum j with current elements"
- **Comparison with Complete Knapsack**: Complete knapsack traverses left to right (allows reuse), 0/1 traverses right to left (no reuse)

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| LC 494 | Target Sum | 0/1 knapsack variant with +/- |
| LC 1049 | Last Stone Weight II | 0/1 knapsack minimization |
| LC 698 | Partition to K Equal Sum Subsets | Backtracking + DP |
| LC 473 | Matchsticks to Square | Similar partition problem |
| LC 805 | Split Array With Same Average | More complex partition constraint |